In [8]:
import torch
import os
import numpy as np
import random
import pickle
import datetime
import argparse
import pandas as pd
import matplotlib.pyplot as plt
import torch.nn.functional as F

from torch.utils.data import DataLoader, Dataset
from models.DeepSleepSota2D import DeepSleepSota2D
from utils.eval_helper import event_level_analysis
from utils.tools import load_edf_file, save_arousal_xml, load_edf_only
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
from matplotlib.colors import LogNorm

def str2bool(v):
    if isinstance(v, bool):
       return v
    if v.lower() in ('yes','true','t','y','1'):
        return True
    elif v.lower() in ('no','false','f','n','0'):
        return False
    else:
        raise argparse.ArgumentTypeError('Boolean value expected.')

def save_to_xml(edf_path, y, save_path, sfreq=50, base_time=None):
    if base_time is None:
        raw = load_edf_file(
            edf_path, 
            preload=True, 
            resample=100, 
            preset="STAGENET", 
            exclude=True, 
            missing_ch='raise'
        )
        base_time = raw.info['meas_date']
    else:
        base_time = datetime.strptime(base_time, "%Y-%m-%d %H:%M:%S")
    save_arousal_xml(base_time, y, sfreq, save_path, min_duration=3, description='AROUS_PRED')

def postprocess_arousal_preds(preds, min_len=5, fs=50):
    """
    특정 길이(min_len) 미만인 이벤트는 제거하는 후처리 함수.
    """
    min_event_samples = int(min_len * fs)
    
    new_preds = np.zeros_like(preds, dtype=int)
    
    in_event = False
    start_idx = 0
    length = len(preds)

    for i in range(length):
        if not in_event:
            if preds[i] == 1:
                in_event = True
                start_idx = i
        else:
            if preds[i] == 0 or i == length - 1:
                if preds[i] == 0:
                    end_idx = i - 1
                else:
                    end_idx = i
                event_len = end_idx - start_idx + 1
                
                if event_len >= min_event_samples:
                    new_preds[start_idx: end_idx + 1] = 1
                in_event = False

    return new_preds

class SpecArousalDataset(Dataset):
    """
    각 pickle 파일에는
      - 'x': shape (9, freq, time)
      - 'y': shape (time,)
    """
    def __init__(self, file_paths, normalize=False):
        super().__init__()
        self.file_paths = file_paths
        self.normalize = normalize
    
    def __len__(self):
        return len(self.file_paths)
    
    def __getitem__(self, idx):
        path = self.file_paths[idx]
        with open(path, 'rb') as f:
            data_dict = pickle.load(f)
        x = data_dict['x']  # shape: (9, freq, time)
        y = data_dict['y']  # shape: (time,)

        info = {
            'freqs': data_dict['freqs'],
            'times': data_dict['times'],
            'y_time': data_dict['y_time'],
            'total_samples': len(data_dict['y_time'])
        }

        x = torch.from_numpy(x)  # (9, freq, time)
        y = torch.from_numpy(y)  # (time,)

        if self.normalize:
            x = (x - x.mean()) / x.std()

        return x, y, info, idx

def map_spec_pred_to_time(
    pred_1d,        
    times,          
    total_samples,  
    fs=50,          
    nperseg=50,     
    mode='average'
):
    half_win_sec = nperseg / (2.0 * fs)  # 예: 2초 윈도우라면 중심으로 +/-1초
    
    y_time = np.zeros(total_samples, dtype=np.float32)
    count  = np.zeros(total_samples, dtype=np.float32) 

    time_bins = len(times)
    for i in range(time_bins):
        center_sec = times[i]
        start_sec = center_sec - half_win_sec
        end_sec   = center_sec + half_win_sec
        
        start_idx = int(np.floor(start_sec * fs))
        end_idx   = int(np.ceil(end_sec * fs))
        
        if start_idx < 0:
            start_idx = 0
        if end_idx > total_samples:
            end_idx = total_samples

        if start_idx >= end_idx:
            continue
        
        if mode == 'average':
            y_time[start_idx:end_idx] += pred_1d[i]
            count[start_idx:end_idx]  += 1.0
        elif mode == 'max':
            y_time[start_idx:end_idx] = np.maximum(y_time[start_idx:end_idx], pred_1d[i])

    if mode == 'average':
        nonzero_mask = (count > 0)
        y_time[nonzero_mask] /= count[nonzero_mask]

    return y_time

def spec_collate_fn(batch_list):
    max_time = 0
    freq_dim = 0
    for (x, y, info, idx) in batch_list:
        _, f, t = x.shape
        freq_dim = f
        if t > max_time:
            max_time = t
   
    batch_size = len(batch_list)
    
    x_batch = torch.zeros(batch_size, 9, freq_dim, max_time, dtype=torch.float)
    y_batch = torch.zeros(batch_size, max_time, dtype=torch.float) + -1  # -1로 padding
    
    idx_list = []
    info_list = []
    
    for i, (x, y, info, idx) in enumerate(batch_list):
        c, f, t = x.shape
        x_batch[i, :, :, :t] = x
        y_batch[i, :t] = y
        idx_list.append(idx)
        info_list.append(info)
    
    idx_tensor = torch.LongTensor(idx_list)
    
    return x_batch, y_batch, info_list, idx_tensor

def eval_fn2(model, loader, device, th=0.923):
    model.eval()
    all_y_pred = []
    all_y_target = []
    all_y_prob = []

    acc, precision, recall, f1 = 0, 0, 0, 0
    n_data = 0

    with torch.no_grad():
        for x, y, info, idx in loader:
            x = x.to(device)
            y = y.to(device)
            
            # forward
            y_pred_2d = model(x)  # (B,1,freq,T_max), 이미 forward에서 sigmoid라면 수정
            # freq pooling -> (B,1,T_max)
            y_pred_1d = y_pred_2d.mean(dim=2)  # or .max(dim=2)[0]
            
            # padding mask
            pad_mask = (y != -1)
            y_pred_1d = y_pred_1d.squeeze(1)
            
            # -1 부분은 실제 데이터가 없으므로 0 처리
            y_pred_1d[~pad_mask] = 0.0
            y[~pad_mask] = 0

            for i, single_idx in enumerate(idx):
                info_i = info[i]
                times = info_i['times']
                total_samples = info_i['total_samples']
                y_target = info_i['y_time']  # (time,) binary

                valid_idx = pad_mask[i]  
                y_pred_i = y_pred_1d[i][valid_idx].cpu().numpy()

                # 시계열 전체에 매핑
                y_pred_logit_time = map_spec_pred_to_time(
                    y_pred_i, times, total_samples, fs=50, nperseg=50
                )
                # threshold
                y_pred_bin = (y_pred_i > th).astype(int)
                y_pred_bin = map_spec_pred_to_time(y_pred_bin, times, total_samples, fs=50, nperseg=50)
                y_pred_bin = (y_pred_bin > 0.5).astype(int)

                acc += accuracy_score(y_target, y_pred_bin)
                precision += precision_score(y_target, y_pred_bin, zero_division=0)
                recall += recall_score(y_target, y_pred_bin, zero_division=0)
                f1 += f1_score(y_target, y_pred_bin, zero_division=0)
                n_data += 1

                # 누적 저장(이후 이벤트 레벨 분석용)
                all_y_pred.append(y_pred_bin)
                all_y_target.append(y_target)
                all_y_prob.append(
                    y_pred_logit_time  # 시계열 점수
                )

    if n_data > 0:
        acc /= n_data
        precision /= n_data
        recall /= n_data
        f1 /= n_data

    return all_y_pred, all_y_target, all_y_prob, acc, precision, recall, f1

class GradCamHook:
    def __init__(self, module):
        self.module = module
        self.hook_f = None
        self.hook_b = None
        self.grad = None
        self.activation = None

    def _forward_hook(self, module, input, output):
        self.activation = output

    def _backward_hook(self, module, grad_in, grad_out):
        self.grad = grad_out[0]

    def register(self):
        self.hook_f = self.module.register_forward_hook(self._forward_hook)
        self.hook_b = self.module.register_backward_hook(self._backward_hook)

    def remove(self):
        self.hook_f.remove()
        self.hook_b.remove()

def generate_gradcam(model, x, target_layer, device, target_idx=None):
    """
    모델에 x(배치=1)를 넣고, target_layer에 대해 Grad-CAM(크기: (H,W))을 계산하여 numpy 배열로 반환.
    """
    hook = GradCamHook(target_layer)
    hook.register()

    model.eval()
    x = x.to(device)
    output = model(x)  # 예) (1,1,51,26500), 이미 sigmoid 됨

    if target_idx is None:
        score = output.sum()
    else:
        fi, ti = target_idx
        score = output[0, 0, fi, ti]

    model.zero_grad()
    score.backward()

    activation = hook.activation  # (1, C, H, W)
    grad = hook.grad             # (1, C, H, W)
    hook.remove()
    
    # Channel-wise gradcam -> (C, H, W)
    activation = activation[0]  # Remove batch dimension (C, H, W)
    grad = grad[0]              # Remove batch dimension (C, H, W)

    # Global Average Pooling (GAP) over spatial dimensions (H, W)
    weights = grad.mean(dim=(1, 2))  # Shape: (C,)

    gradcam = torch.zeros_like(activation[0]) # 
    for c in range(activation.shape[0]): 
        gradcam += weights[c] * activation[c]

    gradcam = F.relu(gradcam)

    gradcam -= gradcam.min()
    gradcam /= gradcam.max() + 1e-8

    return gradcam.detach().cpu().numpy() 
    


def get_time_intervals_from_mask(mask_1d):
    intervals = []
    in_event = False
    start_i = 0
    for i in range(len(mask_1d)):
        if not in_event and mask_1d[i] == 1:
            in_event = True
            start_i = i
        elif in_event and mask_1d[i] == 0:
            intervals.append((start_i, i))  # i-1까지가 1
            in_event = False
    # 끝까지 1인 경우
    if in_event:
        intervals.append((start_i, len(mask_1d)))
    return intervals


# def visualize_gradcam(
#     model,
#     loader,
#     device,
#     target_layer,
#     y_target=None
# ):
#     x_batch, y_batch, info_list, idx_list = next(iter(loader))
#     x_single = x_batch[0:1].to(device)  # shape: (1,9,freq_in,time_in)
#     y_target = y_batch[0:1].cpu().squeeze().numpy()  # shape: (1,time_in)

#     B, C, freq_in, time_in = x_single.shape  # (B=1, C=9, freq_in, time_in)
#     # freq, time = info_list[0]['freqs'], info_list[0]['times']

#     # --------------------
#     # 2) Grad-CAM 생성
#     # --------------------
#     gradcam_map = generate_gradcam(model, x_single, target_layer, device, target_idx=None)
#     print(f"Input shape: {x_single.shape}")
#     print(f"Grad-CAM shape: {gradcam_map.shape}")
#     # gradcam_map: (freq_in, time_in) (UNet 구조상 보통 입력과 동일 크기)

#     # --------------------
#     # 3) 스펙트럼 9채널
#     # --------------------
#     spec_9ch = x_single[0].detach().cpu().numpy()  # shape: (9, freq_in, time_in)

#     # --------------------
#     # 4) y_target -> 빨간색 오버레이 구간(샘플 인덱스)
#     # --------------------
#     intervals_y = []
#     if y_target is not None:
#         # y_target shape=(time_in,) 이라 가정
#         intervals_y = get_time_intervals_from_mask(y_target.astype(int))
#         # intervals_y: [(start_idx, end_idx), ...]

#     # --------------------
#     # 5) Figure (9x1)
#     # --------------------
#     fig, axes = plt.subplots(nrows=9, ncols=1, figsize=(16, 24))

#     # pcolormesh용 meshgrid (x=times, y=freqs)
#     # 여기서는 x=0..time_in, y=0..freq_in 범위를 단순히 인덱스로 사용
#     X, Y = np.meshgrid(np.arange(time_in), np.arange(freq_in))

#     for i in range(9):
#         ax = axes[i]

#         # (A) 스펙트럼 (jet)
#         # shape: (freq_in, time_in)
#         pcm_spec = ax.pcolormesh(
#             X, Y, spec_9ch[i],
#             shading='gouraud',
#             norm=LogNorm(vmin=1e-5, vmax=spec_9ch[i].max()),
#             cmap='jet',
#             alpha=0.2

#         )

#         # # (B) Grad-CAM (magma), alpha=0.5
#         ax.pcolormesh(
#             X, Y, gradcam_map,
#             shading='gouraud',
#             cmap='magma',
#             norm=LogNorm(vmin=0.1, vmax=spec_9ch[i].max()),
#             alpha=1.0
#         )

#         # (C) 정답 구간 => 세로 띠 (x축: [st, en], y축: 전역)
#         #   axvspan(st, en)는 x축을 [st, en]로 채우고, y=전체 범위(0..1) 비율.
#         #   여기서는 axvspan에 x축 단위를 [0..time_in]로 보고,
#         #   y축은 0~freq_in까지 쓰므로, ymin=0, ymax=1(비율)
#         for (st, en) in intervals_y:
#             ax.axvspan(st, en, color='red', alpha=0.5)

#         ax.set_xlim(0, time_in)
#         ax.set_ylim(0, freq_in)
#         ax.set_ylabel(f"Freq(bin)")
#         ax.set_title(f"Channel {i+1}")

#     axes[-1].set_xlabel("Time (sample index)")
#     plt.suptitle("9x1 Spectrogram + Grad-CAM + Arousal(RED)", fontsize=16)
#     plt.tight_layout()
#     plt.show()

def visualize_gradcam(
    model,
    loader,
    device,
    target_layer,
    y_target=None,
    num_segments=5
):
    # 1) 샘플 하나 가져오기
    x_batch, y_batch, info_list, idx_list = next(iter(loader))
    x_single = x_batch[0:1].to(device)        # (1,9,freq,time)
    y_target = y_batch[0].cpu().numpy()       # (time,)
    # y_pred = model(x_single)

    # 2) Grad‑CAM 계산
    gradcam_map = generate_gradcam(model, x_single, target_layer, device)
    freq_in, time_in = gradcam_map.shape      # (freq, time)

    # 3) 스펙트로그램 채널별 추출
    spec_9ch = x_single[0].cpu().numpy()      # (9, freq, time)

    # 4) 이벤트 구간 리스트
    intervals_y = []
    if y_target is not None:
        intervals_y = get_time_intervals_from_mask(y_target.astype(int))

    # intervals_y_pred = get_time_intervals_from_mask(y_pred[0].detach().cpu().numpy().astype(int))

    # 5) 5등분 세그먼트 계산
    segment_length = time_in // num_segments
    segments = [
        (i*segment_length, (i+1)*segment_length if i< num_segments-1 else time_in)
        for i in range(num_segments)
    ]

    # 6) 각 세그먼트별로 그림
    for seg_idx, (start, end) in enumerate(segments):
        fig, axes = plt.subplots(nrows=9, ncols=1, figsize=(16, 24))
        for ch in range(9):
            ax = axes[ch]
            spec = spec_9ch[ch][:, start:end]
            cam  = gradcam_map[:,   start:end]

            # (A) Spectrogram: 완전 불투명
            ax.imshow(
                spec,
                aspect='auto',
                origin='lower',
                norm=LogNorm(vmin=spec.min()+1e-5, vmax=spec.max()),
                cmap='jet',
                alpha=0.7,
                extent=[start, end, 0, freq_in]
            )
            # (B) Grad‑CAM: 반투명 오버레이
            ax.imshow(
                cam,
                aspect='auto',
                origin='lower',
                norm=LogNorm(vmin=spec.min()+1e-5, vmax=spec.max()),
                cmap='magma',
                alpha=1.0,
                extent=[start, end, 0, freq_in]
            )

            # (C) 정답 이벤트 구간 표시
            for (st, en) in intervals_y:
                if en <= start or st >= end:
                    continue
                ov_st, ov_en = max(st, start), min(en, end)
                ax.axvspan(ov_st, ov_en, color='red', alpha=0.2)

            # for (st, en) in intervals_y_pred:
            #     if en <= start or st >= end:
            #         continue
            #     ov_st, ov_en = max(st, start), min(en, end)
            #     ax.axvspan(ov_st, ov_en, color='black', alpha=0.4)

            ax.set_xlim(start, end)
            ax.set_ylim(0, freq_in)
            ax.set_ylabel(f"Ch {ch+1}")
        axes[-1].set_xlabel("Time (sample idx)")
        plt.suptitle(f"Segment {seg_idx+1}: samples {start}–{end}", fontsize=16)
        plt.tight_layout()
        plt.show()


def visualize_gradcam_old(model, loader, device, target_layer):
    """
    val_loader(또는 test_loader)에서 배치를 하나 꺼낸 뒤, Grad-CAM heatmap을 구해
    (9,51,26500) 스펙트럼 각각에 heatmap을 오버레이하여 시각화.
    """
    # 1) loader에서 배치 1개만 가져옴
    x_batch, y_batch, info_list, idx_list = next(iter(loader))
    # x_batch shape: (B, 9, 51,  ~time~ )
    # 여기서는 time=26500이지만, dataset에 따라 달라질 수 있음
    x_single = x_batch[0:1]  # 첫 번째 샘플만 (shape: (1,9,51,26500))

    # 2) Grad-CAM 계산
    gradcam_map = generate_gradcam(model, x_single, target_layer, device, target_idx=None)
    # gradcam_map: (H, W) = (51, 26500) (UNet 구조상, target_layer 출력과 동일한 spacial size)

    freq_in = x_single.shape[2]  # 51
    time_in = x_single.shape[3]  # 26500

    freq_cam, time_cam = gradcam_map.shape
    if (freq_in != freq_cam) or (time_in != time_cam):
        # 크기가 다르면 보간
        gradcam_tensor = torch.from_numpy(gradcam_map).unsqueeze(0).unsqueeze(0)  # (1,1,H_cam,W_cam)
        gradcam_resized = F.interpolate(
            gradcam_tensor, 
            size=(freq_in, time_in), 
            mode='bilinear', 
            align_corners=False
        )
        gradcam_map = gradcam_resized.squeeze().numpy()  # (freq_in, time_in)

    # 3) 원본 9채널 스펙트럼 (9,51,26500) 꺼내기
    spec_9ch = x_single[0].cpu().numpy()

    # 4) 9채널 각각에 Grad-CAM overlay 시각화
    fig, axes = plt.subplots(3, 3, figsize=(15, 10))
    axes = axes.flatten()

    for i in range(9):
        ax = axes[i]
        # i번째 채널 스펙 (shape: (51, 26500))
        spec_ch = spec_9ch[i]

        # 원본 스펙트럼 시각화
        # freq축(세로) = 51, time축(가로) = 26500 → 매우 큼
        im = ax.imshow(spec_ch, aspect='auto', origin='lower', cmap='jet')

        # Grad-CAM heatmap 오버레이
        ax.imshow(gradcam_map, aspect='auto', origin='lower', cmap='magma', alpha=0.4)

        ax.set_title(f'Channel {i+1}')
        ax.axis('off')

    fig.suptitle('Grad-CAM Visualization on 9-Channel Spectrogram')
    plt.tight_layout()
    plt.show()


# -------------------- 메인 함수 --------------------
def main(edf_path, save_path=None):
    torch.manual_seed(42)
    np.random.seed(42)
    random.seed(42)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

    device = f'cuda:0' if torch.cuda.is_available() else 'cpu'

    # (예시) arousal_dir는 실제 상황에 맞게 수정
    arousal_dir = os.path.dirname(edf_path).replace("EDF", "AROUS_SPEC")
    arousal_dir = "/home/honeynaps/data/GOLDEN/AROUS_SPEC"
    test_dir = f"{arousal_dir}/AROUSAL_SPEC_50_PAD_tight"

    edf_name = os.path.basename(edf_path)
    val_files = [os.path.join(test_dir, edf_name.replace(".edf", ".pkl"))]

    val_dataset  = SpecArousalDataset(val_files)
    val_loader   = DataLoader(val_dataset,
                              batch_size=1,
                              shuffle=False,
                              num_workers=1,
                              collate_fn=spec_collate_fn)

    # 모델 생성
    model = DeepSleepSota2D(in_channels=9).to(device)
    
    # 예시로 pretrained_path 지정 (사용 환경에 따라 수정)
    pretrained_path = "/home/honeynaps/data/saved_models_spec/ChunkSpecW2__f1_0.8007__PAD_tight_lr0.0010_fs50_ep17_auprc0.8471_th0.2412.pt"
    th = float(pretrained_path.split('_')[-1].replace('.pt', '').replace('th', ''))
    
    # 모델 로드
    state_dict = torch.load(pretrained_path, map_location=device)
    # state_dict가 'weights_only=True'로 저장되었는지 여부에 맞춰서 키를 맞춰야 함
    if isinstance(state_dict, dict) and 'model_state_dict' in state_dict:
        state_dict = state_dict['model_state_dict']
    model.load_state_dict(state_dict)

    # 추론(샘플 평가)
    y_preds, y_targets, y_probs, acc, precision, recall, fl = eval_fn2(model, val_loader, device, th=th)
    print(f"[Before Postprocess]\nAccuracy: {acc:.4f}, Precision: {precision:.4f}, Recall: {recall:.4f}, F1: {fl:.4f}")

    # 후처리 및 이벤트 레벨 분석 (단일 파일이므로 y_preds[0], y_targets[0], y_probs[0]만 존재한다고 가정)
    y_pred = y_preds[0]
    y_target = y_targets[0]
    y_prob = y_probs[0]

    # 예: 길이 3.8초 미만의 이벤트는 제거
    y_pred = postprocess_arousal_preds(y_pred, min_len=3.8, fs=50)

    print("[After Postprocess]")
    acc2 = accuracy_score(y_target, y_pred)
    precision2 = precision_score(y_target, y_pred, zero_division=0)
    recall2 = recall_score(y_target, y_pred, zero_division=0)
    f12 = f1_score(y_target, y_pred, zero_division=0)
    print(f"Accuracy: {acc2:.4f}, Precision: {precision2:.4f}, Recall: {recall2:.4f}, F1: {f12:.4f}")


    target_layer = model.out_conv
    
    # val_loader 재사용해서 Grad-CAM 시각화
    visualize_gradcam(model, val_loader, device, target_layer)

    return acc2, precision2, recall2, f12



In [9]:
edf_path = "/home/honeynaps/data/GOLDEN/EDF2/SCH_M_30_OB_230617R3_SE.edf"
save_path = '/home/honeynaps/data/shared/arousal'
main(edf_path, save_path)

[Before Postprocess]
Accuracy: 0.9389, Precision: 0.8422, Recall: 0.8409, F1: 0.8415
[After Postprocess]
Accuracy: 0.9388, Precision: 0.8437, Recall: 0.8376, F1: 0.8406


ValueError: The truth value of an array with more than one element is ambiguous. Use a.any() or a.all()